In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

import xgboost as xgb

In [2]:
df = pd.read_csv('../data/car_fuel_efficiency_2026.csv')

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(10000, 11)
model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64


In [3]:
df = df.fillna(0)
df.isnull().sum().sum()   # should print 0

np.int64(0)

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [5]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

features = list(dv.get_feature_names_out())
features

['acceleration',
 'drivetrain=All-wheel drive',
 'drivetrain=Front-wheel drive',
 'drivetrain=Rear-wheel drive',
 'engine_displacement',
 'fuel_type=Diesel',
 'fuel_type=Gasoline',
 'fuel_type=Hybrid',
 'horsepower',
 'model_year',
 'num_cylinders',
 'num_doors',
 'origin=Asia',
 'origin=Europe',
 'origin=USA',
 'vehicle_weight']

In [6]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=features))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [7]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)

1.837054054730018

In [8]:
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    scores.append((n, round(rmse, 3)))

pd.DataFrame(scores, columns=['n_estimators', 'rmse'])

,n_estimators,rmse
0,10,1.837
1,50,1.772
2,100,1.768
3,150,1.769


In [9]:
scores = []

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        scores.append((d, n, rmse))

df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
df_scores.groupby('max_depth').rmse.mean()

max_depth
10    1.755405
15    1.783953
20    1.788242
25    1.786317
Name: rmse, dtype: float64

In [10]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

pd.DataFrame({'feature': features, 'importance': rf.feature_importances_}) \
    .sort_values('importance', ascending=False)

,feature,importance
9,model_year,0.296001
15,vehicle_weight,0.203879
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201
14,origin=USA,0.042144
2,drivetrain=Front-wheel drive,0.020145
11,num_doors,0.019415
5,fuel_type=Diesel,0.017038


In [11]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

def train_xgb(eta):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    evals_result = {}
    xgb.train(xgb_params, dtrain, num_boost_round=100,
              evals=watchlist, evals_result=evals_result,
              verbose_eval=False)
    return evals_result['val']['rmse']

for eta in [0.3, 0.1]:
    val_rmse = train_xgb(eta)
    print(f'eta={eta}: RMSE after 100 rounds = {val_rmse[-1]:.4f}, '
          f'best = {min(val_rmse):.4f}')

eta=0.3: RMSE after 100 rounds = 1.8265, best = 1.7320
eta=0.1: RMSE after 100 rounds = 1.7248, best = 1.7122
